## Data Load in

In [ ]:
# you will be prompted with a window asking to grant permissions
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
# fill in the path in your Google Drive in the string below. Note: do not escape slashes or spaces
import os
datadir = "/content/drive/Shareddrives/CS444 DLCV 抱團/Final Project/"
if not os.path.exists(datadir):
  !ln -s "/content/drive/Shareddrives/CS444 DLCV 抱團/Final Project/" $datadir # TODO: Fill your assignment3 path
os.chdir(datadir)
!pwd

/content/drive/Shareddrives/CS444 DLCV 抱團/Final Project


In [ ]:

!unzip "/content/drive/Shareddrives/CS444 DLCV 抱團/Final Project/test.zip" -d "/content/data2"


串流輸出內容已截斷至最後 5000 行。
  inflating: /content/data2/test/cc358864-bacb-11e8-b2b8-ac1f6b6435d0_blue.png  
  inflating: /content/data2/__MACOSX/test/._cc358864-bacb-11e8-b2b8-ac1f6b6435d0_blue.png  
  inflating: /content/data2/test/bfd7cda4-bac9-11e8-b2b8-ac1f6b6435d0_yellow.png  
  inflating: /content/data2/__MACOSX/test/._bfd7cda4-bac9-11e8-b2b8-ac1f6b6435d0_yellow.png  
  inflating: /content/data2/test/b029f372-bad9-11e8-b2b9-ac1f6b6435d0_yellow.png  
  inflating: /content/data2/__MACOSX/test/._b029f372-bad9-11e8-b2b9-ac1f6b6435d0_yellow.png  
  inflating: /content/data2/test/7a0c7fd4-bac6-11e8-b2b7-ac1f6b6435d0_yellow.png  
  inflating: /content/data2/__MACOSX/test/._7a0c7fd4-bac6-11e8-b2b7-ac1f6b6435d0_yellow.png  
  inflating: /content/data2/test/404e60b2-bac8-11e8-b2b7-ac1f6b6435d0_green.png  
  inflating: /content/data2/__MACOSX/test/._404e60b2-bac8-11e8-b2b7-ac1f6b6435d0_green.png  
  inflating: /content/data2/test/9fa35cce-bac9-11e8-b2b8-ac1f6b6435d0_green.png  
  inflating: /cont

In [ ]:
import pandas as pd

df = pd.read_csv("/content/data2/sample_submission.csv")
df.head()

,Id,Predicted
0,00008af0-bad0-11e8-b2b8-ac1f6b6435d0,0
1,0000a892-bacf-11e8-b2b8-ac1f6b6435d0,0
2,0006faa6-bac7-11e8-b2b7-ac1f6b6435d0,0
3,0008baca-bad7-11e8-b2b9-ac1f6b6435d0,0
4,000cce7e-bad4-11e8-b2b8-ac1f6b6435d0,0


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class MultiLabelArcFaceLoss(nn.Module):
    def __init__(self, s=30.0, m=0.5, reduction='mean'):
        super(MultiLabelArcFaceLoss, self).__init__()
        self.s = s
        self.m = m
        self.reduction = reduction

        self.cos_m = math.cos(m)
        self.sin_m = math.sin(m)
        self.th = math.cos(math.pi - m)
        self.mm = math.sin(math.pi - m) * m

        self.bce_loss = nn.BCEWithLogitsLoss(reduction=reduction)

    def forward(self, logits, labels):
        # logits: [B, C] - cosine similarities before scaling
        # labels: [B, C] - multi-hot vectors (0 or 1)

        cosine = logits.clamp(-1.0 + 1e-7, 1.0 - 1e-7)  # 防止 sqrt 負數
        sine = torch.sqrt(1.0 - torch.pow(cosine, 2))

        phi = cosine * self.cos_m - sine * self.sin_m
        phi = torch.where(cosine > self.th, phi, cosine - self.mm)

        # 混合 φ 和原 cosine：對於正類使用 phi，對於負類保持 cosine
        output = torch.where(labels.bool(), phi, cosine)

        # 放大
        output *= self.s

        loss = self.bce_loss(output, labels.float())
        return loss


import torch.nn.functional as F
from torch.nn.parameter import Parameter

class ArcMarginProduct(nn.Module):
    def __init__(self, in_features, out_features):
        super(ArcMarginProduct, self).__init__()
        self.weight = Parameter(torch.FloatTensor(out_features, in_features))
        self.reset_parameters()

    def reset_parameters(self):
        stdv = 1. / math.sqrt(self.weight.size(1))
        self.weight.data.uniform_(-stdv, stdv)

    def forward(self, features):
        cosine = F.linear(F.normalize(features), F.normalize(self.weight))
        return cosine

class MyMultiLabelArcFaceModel(nn.Module):
    def __init__(self, base_model, num_classes):
        super().__init__()
        self.extract_feature = False
        self.EX = 2

        self.backbone = base_model
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.maxpool = nn.AdaptiveMaxPool2d(1)

        self.bn1 = nn.BatchNorm1d(2048 * self.EX)  # = 4096
        self.fc1 = nn.Linear(2048 * self.EX, 512 * self.EX)  # e.g. 4096 -> 1024
        self.bn2 = nn.BatchNorm1d(512 * self.EX)
        self.relu = nn.ReLU(inplace=True)
        self.fc2 = nn.Linear(512 * self.EX, 512)
        self.bn3 = nn.BatchNorm1d(512)

        self.arc_margin_product = ArcMarginProduct(512, num_classes)

    def forward(self, x):
        e5 = self.backbone(x)  # e.g., ResNet's output before fc
        x = torch.cat((self.avgpool(e5), self.maxpool(e5)), dim=1)  # [B, 4096, 1, 1]
        x = x.view(x.size(0), -1)  # [B, 4096]
        x = self.bn1(x)
        x = F.dropout(x, p=0.25)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.bn2(x)
        x = F.dropout(x, p=0.5)

        x = self.fc2(x)
        feature = self.bn3(x)

        cosine = self.arc_margin_product(feature)
        if self.extract_feature:
            return cosine, feature
        else:
            return cosine




In [ ]:
import torch
from torchvision import transforms
from PIL import Image
import pandas as pd


model = torch.load('/content/drive/Shareddrives/CS444 DLCV 抱團/Final Project/model_resnet18_focalLoss_epoch30.pth', weights_only=False)
model.eval()


df = pd.read_csv("/content/data2/sample_submission.csv")


transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
])


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

colors = ['red', 'green', 'blue', 'yellow']
threshold = 0.5
all_preds = []

for img_id in df['Id']:

    channels = []
    for color in colors:
        img_path = f"/content/data2/test/{img_id}_{color}.png"
        image = Image.open(img_path).convert('L')
        image = transform(image)
        channels.append(image)

    image_4ch = torch.cat(channels, dim=0).unsqueeze(0).to(device)  # (1, 4, H, W)

    with torch.no_grad():
        output = model(image_4ch)  # (1, C)
        probs = torch.sigmoid(output)
        pred_onehot = (probs > threshold).squeeze(0).cpu().int()  # (C,)


        if pred_onehot.sum() == 0:

            top1 = torch.argmax(probs, dim=1).item()
            pred_onehot[top1] = 1


        pred_indices = [str(i) for i, val in enumerate(pred_onehot) if val == 1]
        pred_str = " ".join(pred_indices)
        all_preds.append(pred_str)



df['Predicted'] = all_preds
df.to_csv("/content/drive/Shareddrives/CS444 DLCV 抱團/Final Project/submission_resnet18_focalLoss_epoch30.csv", index=False)
print("✅ Submission saved to /content/submission.csv")


✅ Submission saved to /content/submission.csv
